In [1]:
colab = False

In [ ]:
%load_ext autoreload
%autoreload 2
if colab:
    from google.colab import drive
    drive.mount('/content/drive')
    %cd drive/MyDrive/ecg_arrhythmia/
    !pip install wfdb wget numpy pandas scipy scikit-learn tensorflow matplotlib seaborn PyWavelets
    !pip install --upgrade wfdb
    !pip install neurokit2
from src.models.train_model import split_data, compute_class_weights, one_hot_encode, train_and_evaluate
from src.preprocess.pipeline import run_pipeline
from src.utils.utils import export_hyperparams
from config import  HYPERPARAMS, GRIDSEARCH_PATH, TEST_PATH
import os
import pandas as pd
import numpy as np

path_list = [GRIDSEARCH_PATH, TEST_PATH]
for path in path_list:
    if not os.path.exists(path):
        os.makedirs(path)
        
resfile_name = 'splitseed0-39_withgroup_classweight_x1only.csv' 

In [ ]:
x1, x2, y, records = run_pipeline(HYPERPARAMS['model_type'], HYPERPARAMS['hrv_window'])

print("Segments(x1) Shape:", x1.shape)
if HYPERPARAMS['model_type'] == 1:
    print("Extracted Features(x2) Shape:", x2.shape)
print("Labels(y) Shape:", y.shape)

In [14]:
# ## plotting
# label_hist(y)
# label_record_hist(y, records)
# # get_label_record_ratio(y, records) # label별 record 비율 desc 으로 출력

In [ ]:
all_metrics = []
def train_test(seed, model_type, x1, x2, y, records):
    train, val ,test = split_data(x1, y, records, seed, x2)
    # 클래스 가중치 계산 (학습 데이터 기준)
    class_weights = compute_class_weights(train['y'])
    # one-hot encoding
    y_train_oh, y_val_oh, y_test_oh, class_names = one_hot_encode(train['y'], val['y'], test['y'])
    # 모델 학습 및 평가
    model, metric_df = train_and_evaluate(model_type, train, val, test,
                    y_train_oh, y_val_oh,y_test_oh, class_names, class_weights, seed)
    # 결과 저장 및 시각화
    all_metrics.append(metric_df)
    metric_df.to_csv(GRIDSEARCH_PATH + f"metrics_seed{seed}.csv", index=False)
    # 모델 예측 결과에 대한 평가 지표 재계산 후 히스토그램 출력
    
    

In [ ]:
def train_test(x1, x2, y, records):
    seed = HYPERPARAMS['seed']
    model_type = HYPERPARAMS['model_type']
    res_dir = f"res/seed{seed}_model{model_type}"
    os.makedirs(res_dir, exist_ok=True)
    train, val ,test = split_data(x1, y, records, seed, x2)
    class_weights = compute_class_weights(train['y'])
    y_train_oh, y_val_oh, y_test_oh, class_names = one_hot_encode(train['y'], val['y'], test['y'])
    model, metric_df = train_and_evaluate(model_type, train, val, test,
                    y_train_oh, y_val_oh, y_test_oh, class_names, class_weights, seed)
    all_metrics.append(metric_df)
    # Save metrics
    metric_df.to_csv(os.path.join(res_dir, "metrics.csv"), index=False)
    # Save predictions
    if model_type == 1:
        y_pred = model.predict([test['x1'], test['x2']])
    else:
        y_pred = model.predict(test['x1'])
    pd.DataFrame({
        "y_true": list(np.argmax(y_test_oh, axis=1)),
        "y_pred": list(y_pred)
    }).to_csv(os.path.join(res_dir, "predictions.csv"), index=False)
    # Save hyperparams
    export_hyperparams(HYPERPARAMS, os.path.join(res_dir, "hyperparams.json"))

In [ ]:
train_test(HYPERPARAMS['seed'], HYPERPARAMS['model_type'], x1, x2, y, records)